<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Step 7</span>
<h1 style="font-size:1.35em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Step 7: Surviving dead connections</h1>
<p style="margin:.55em 0">Measured in practice: about one long reply in six, OpenAI lets a connection go silent mid-response and stop answering
pings, while new connections work fine. The watchdog (step 6) detects it in seconds. The fix is architectural: since the
session's state is ours, open a new connection, <b>replay</b> the words and tool calls so far plus our state, and carry on.
The caller's audio is pumped through a queue so nothing they say during the switch is lost.</p>

</div>

In [1]:
%%writefile -a build/app/realtime.py

async def converse_resilient(start, chunks, speaker, seconds: float, max_resumes: int = 2, on_note=None,
                             stop: asyncio.Event | None = None, context=None, **kw) -> list[tuple]:
    """converse() that survives a dead OpenAI connection.

    start: async () -> a new configured session. On a dead link: open a new one, replay the turns and tool calls
    so far (+ context(), our own state) and ask the model to carry on. At most `max_resumes` reconnects.
    The caller's audio is pumped through a queue, so nothing said during a reconnect is lost."""
    q: asyncio.Queue = asyncio.Queue()
    history, log_, t0 = [], [], time.perf_counter()

    async def pump():  # the one reader of `chunks`; its end or error is queued for whichever attempt is current
        try:
            async for c in chunks:
                await q.put(c)
            await q.put(StopAsyncIteration())
        except Exception as e:  # e.g. the browser left: must end the conversation, not hang it
            await q.put(e)

    async def from_queue():
        while True:
            c = await q.get()
            if isinstance(c, BaseException):
                q.put_nowait(c)  # keep it: a later attempt must see it too
                if isinstance(c, StopAsyncIteration):
                    return
                raise c
            yield c

    def note(t, event, detail):
        log_.append((t, event, detail))
        if event in ("user_said", "model_said"):
            history.append(("user" if event == "user_said" else "assistant", detail))
        elif event == "tool_call":  # replay what already happened, or the model redoes it (e.g. a spent code)
            history.append(("system", f"Already done: {detail}"))
        if on_note:
            on_note(t, event, detail)

    pumper = asyncio.create_task(pump())
    deadline = t0 + seconds
    ws = await start()
    try:
        for attempt in range(max_resumes + 1):
            try:
                await converse(ws, from_queue(), speaker, max(deadline - time.perf_counter(), 0), on_note=note,
                               stop=stop, **kw)
                return log_
            except DROPPED as e:
                if attempt == max_resumes or deadline - time.perf_counter() < 5:
                    raise
                METRICS.inc("resumes_total")
                detail = f"{type(e).__name__}: reconnecting ({attempt + 1}/{max_resumes}), replaying {len(history)} items"
                log.warning("RESUME %s", detail)
                note(round(time.perf_counter() - t0, 2), "RESUME", detail)
                with contextlib.suppress(Exception):
                    ws.transport.abort()  # a dead link never answers a polite close: don't wait 10 s for it
                ws = await start()
                await replay(ws, history, context() if context else "")
                await send(ws, {"type": "response.create"})
        return log_
    finally:
        pumper.cancel()
        with contextlib.suppress(Exception):
            await ws.close()


Appending to build/app/realtime.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Try it: a fake OpenAI that dies mid-reply</h2><p style="margin:.55em 0">The first connection stops reading (so pings go unanswered); the second answers.</p>

</div>

In [2]:
import sys
sys.path.insert(0, 'build')   # `import app` now means the app you are building

In [3]:
import asyncio, json, time
from websockets.asyncio.server import serve
from websockets.asyncio.client import connect
from app import realtime

class Speaker:
    def __init__(self): self.buf = bytearray(); self.played = 0
    position = property(lambda self: self.played + len(self.buf))
    def write(self, b): self.buf += b
    def clear(self): n = len(self.buf); self.buf.clear(); return n
    def buffered_seconds(self): return 0

conns, replayed = [], []

async def fake_openai(ws):
    conns.append(ws)
    if len(conns) == 1:
        await ws.recv()
        for e in ({'type': 'conversation.item.input_audio_transcription.completed', 'transcript': 'Book me in'},
                  {'type': 'response.created', 'response': {}}):
            await ws.send(json.dumps(e))
        ws.transport.pause_reading()   # silent, and pings go unanswered
        await asyncio.sleep(8)
        return
    async for raw in ws:
        m = json.loads(raw)
        if m['type'] == 'conversation.item.create':
            replayed.append((m['item']['role'], m['item']['content'][0]['text'][:60]))
        elif m['type'] == 'response.create':
            for e in ({'type': 'response.created', 'response': {}},
                      {'type': 'response.output_audio_transcript.done', 'transcript': 'Sure, which doctor?'},
                      {'type': 'response.done', 'response': {'status': 'completed', 'output': []}}):
                await ws.send(json.dumps(e))

async def mic():
    while True:
        yield bytes(4800)
        await asyncio.sleep(0.1)

In [4]:
done, t0 = asyncio.Event(), time.monotonic()
async with serve(fake_openai, '127.0.0.1', 8791):
    log = await realtime.converse_resilient(
        lambda: connect('ws://127.0.0.1:8791'), mic(), Speaker(), seconds=30, context=lambda: 'Our state: {...}',
        on_note=lambda t, ev, d: done.set() if ev == 'model_said' else None, stop=done, stall_s=1, ping_timeout_s=2)
[(ev, d[:80]) for _, ev, d in log], f'{time.monotonic() - t0:.1f}s'

RESUME Stalled: reconnecting (1/2), replaying 1 items


([('user_said', 'Book me in'),
  ('response.created', ''),
  ('STALL',
   'no server event for 1.0s mid-response; no pong (TimeoutError); our loop max lag '),
  ('RESUME', 'Stalled: reconnecting (1/2), replaying 1 items'),
  ('response.created', ''),
  ('model_said', 'Sure, which doctor?'),
  ('response.done', 'completed')],
 '13.0s')

In [5]:
replayed   # the words so far, then our system note with the state

[('user', 'Book me in'),
 ('system', 'The connection dropped and was restored. Continue from where')]

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0"><b>Next:</b> <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">08_server.ipynb</code>: putting a browser on the other end.</p>

</div>